# 04 -- OCR baseline (LeNet5 + hard-coded solver)

## 0. Colab bootstrap (no-op locally)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys, os
    PROJECT_ROOT = '/content/drive/MyDrive/Master-thesis'
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT + '/notebooks')
    print('Colab setup done -- cwd:', os.getcwd())
except ImportError:
    print('Local run -- Colab bootstrap skipped')

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'mnlearn').is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('could not locate project root')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

from mnlearn.baselines import train_baseline
from mnlearn.config import load_baseline_config
from mnlearn.experiments import collect_results

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_TAG    = 'ocr_baseline_main'
RESULTS    = REPO_ROOT / 'results' / RUN_TAG
THESIS_FIG = REPO_ROOT / 'thesis' / 'figures' / RUN_TAG
THESIS_TBL = REPO_ROOT / 'thesis' / 'tables'
THESIS_FIG.mkdir(parents=True, exist_ok=True)
THESIS_TBL.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = load_baseline_config(
    REPO_ROOT / 'configs' / 'experiments' / 'baseline_sudoku_ocr.yaml'
)
print(f'DEVICE = {DEVICE}')

## 2. Sweep

N values match notebook 05's learning curve for direct overlay in the thesis writeup.

In [ ]:
N_VALUES = [100, 500, 1000, 1500, 5000]
SEEDS    = [0, 1, 2]

def out_dir(n, seed):
    return RESULTS / f'run_n{n}_seed{seed}'

## 3. Run

In [ ]:
for seed in SEEDS:
    for n in N_VALUES:
        d = out_dir(n, seed)
        if (d / 'results.json').exists():
            print(f'skip: {d.name}')
            continue
        name = f'{RUN_TAG}/{d.name}'
        cfg = replace(
            BASE_CONFIG,
            experiment=replace(BASE_CONFIG.experiment, name=name, seed=seed),
            data=replace(BASE_CONFIG.data, train_size=n),
        )
        print(f'running: {name}')
        train_baseline(cfg)

print('\ndone.')

## 4. Aggregate

In [ ]:
df = collect_results(str(RESULTS / 'run_n*_seed*/'))
df = df[df['n'].isin(N_VALUES) & df['seed'].isin(SEEDS)].copy()
cols = ['n', 'seed', 'zero_one', 'solver_feasibility', 'ocr_clue_error',
        'best_epoch', 'final_epoch', 'total_seconds']
df = df[[c for c in cols if c in df.columns]].sort_values(['n', 'seed']).reset_index(drop=True)
df.to_csv(THESIS_TBL / 's5_3_5_ocr_baseline_sweep.csv', index=False)
df

## 5. Headline table

In [ ]:
metrics = ['zero_one', 'solver_feasibility', 'ocr_clue_error']
agg = (df.groupby('n')[metrics]
         .agg(['mean', 'std', 'count'])
         .reset_index())
agg.columns = ['_'.join([c for c in col if c]).rstrip('_') for col in agg.columns]
print(agg.to_string(index=False))
agg.to_csv(THESIS_TBL / 's5_3_5_ocr_baseline_table.csv', index=False)

## 6. Learning curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panels = [
    ('zero_one',           'test 0/1'),
    ('ocr_clue_error',     'OCR per-clue error'),
    ('solver_feasibility', 'solver feasibility'),
]

for ax, (metric, ylabel) in zip(axes, panels):
    g = (df.groupby('n')[metric]
           .agg(['mean', 'std']).reset_index().sort_values('n'))
    line, = ax.plot(g['n'], g['mean'], label='OCR baseline')
    ax.fill_between(g['n'],
                    g['mean'] - g['std'].fillna(0.0),
                    g['mean'] + g['std'].fillna(0.0),
                    alpha=0.2, color=line.get_color())
    ax.set_xscale('log')
    ax.set_xlabel('N_train')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs N')
    ax.legend()
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle(f'OCR baseline learning curve -- {len(SEEDS)} seeds')
fig.tight_layout()
fig.savefig(THESIS_FIG / 'learning_curve.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. OCR training curve (largest N, seed 0)

In [ ]:
N_BIG = max(N_VALUES)
history = json.loads((out_dir(N_BIG, SEEDS[0]) / 'history.json').read_text())

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(history['epoch'], history['train_error'], linestyle='--', alpha=0.7, label='train error')
ax.plot(history['epoch'], history['val_error'], label='val error')
ax.axvline(history['best_epoch'], color='red', linestyle=':',
           label=f"best epoch ({history['best_epoch']})")
ax.set_xlabel('epoch')
ax.set_ylabel('per-cell classification error')
ax.set_title(f'OCR training curve -- N={N_BIG}, seed {SEEDS[0]}')
ax.legend()
ax.grid(True, alpha=0.3)

fig.savefig(THESIS_FIG / 'training_curve.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Feasibility vs OCR error

Each point is one (N, seed) run. Strong negative correlation means solver failures are explained by OCR errors making puzzles infeasible.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
scatter = ax.scatter(df['ocr_clue_error'], df['solver_feasibility'],
                     c=df['n'], cmap='viridis', s=60, alpha=0.8,
                     edgecolors='black', linewidths=0.5)
for n_val in N_VALUES:
    sub = df[df['n'] == n_val]
    if len(sub) == 0:
        continue
    ax.annotate(f'N={n_val}',
                xy=(sub['ocr_clue_error'].mean(), sub['solver_feasibility'].mean()),
                xytext=(5, 5), textcoords='offset points', fontsize=9, color='darkred')

ax.set_xlabel('OCR per-clue error rate')
ax.set_ylabel('solver feasibility')
ax.set_title('OCR error vs solver feasibility')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('N_train')

fig.savefig(THESIS_FIG / 'feasibility.png', bbox_inches='tight', dpi=150)
plt.show()

corr = df[['ocr_clue_error', 'solver_feasibility']].corr(method='spearman').iloc[0, 1]
print(f'Spearman correlation (ocr_clue_error vs solver_feasibility): {corr:+.3f}')